# Pipeline de Limpieza: Dataset de Películas (2013-2015)
Este Notebook documenta el proceso de ETL para normalizar el dataset de películas, resolviendo problemas de desplazamiento de columnas y tipado de datos.

## 1. Diagnóstico de Datos
Hito: Identificación del error de Column Shifting mediante la carga de datos sin calificadores, exponiendo visualmente el desajuste de las columnas.

In [1]:
import pandas as pd
import csv

# names para crear columnas 0, 1, 2... hasta 24, para que no de error de ParserError: Expected X fields, saw Y por desplazamiento
df_error = pd.read_csv('../data/raw/Movie-Data.csv', 
                          quoting=csv.QUOTE_NONE, 
                          names=range(25)) 

print("Desplazamiento encontrado",df_error.iloc[454].dropna())

Desplazamiento encontrado 0                                   "The Way
1                                  Way Back"
2                                 2013-07-05
3     "https://en.wikipedia.org/wiki/The_Way
4                                 _Way_Back"
5                                     Comedy
6                                  Nat Faxon
7                                   Jim Rash
8                               Steve Carell
9                              Toni Collette
10                            Allison Janney
11                           AnnaSophia Robb
12                              Sam Rockwell
13                                       "$5
14                                       000
15                                   000.00"
16                                       "$5
17                                       0.0
18                                   000.00"
Name: 454, dtype: object


## 2. Ingeniería de Ingesta
Hito: Implementación de parámetros avanzados `quotechar='"'` en Pandas para encapsular los títulos y proteger la integridad de los datos frente a las comas internas.

In [2]:
# Aplico el calificador de texto para que las comas dentro de los títulos no rompan el esquema
df = pd.read_csv('../data/raw/Movie-Data.csv', quotechar='"')

print("Dataset cargado correctamente con dimensiones:", df.shape)

# Muestro la fila que presentaba desplazamiento, ahora se muestra correctamente
df.iloc[453]

Dataset cargado correctamente con dimensiones: (508, 13)


Movie Title                                    The Way, Way Back
Release Date                                          2013-07-05
Wikipedia URL    https://en.wikipedia.org/wiki/The_Way,_Way_Back
Genre                                                     Comedy
Director (1)                                           Nat Faxon
Director (2)                                            Jim Rash
Cast (1)                                            Steve Carell
Cast (2)                                           Toni Collette
Cast (3)                                          Allison Janney
Cast (4)                                         AnnaSophia Robb
Cast (5)                                            Sam Rockwell
Budget                                             $5,000,000.00
Revenue                                            $5,000,000.00
Name: 453, dtype: object

## 3. Normalización de Esquema
Hito: Transformación de encabezados (eliminación de paréntesis y espacios) hacia una convención snake_case, garantizando la compatibilidad técnica con motores SQL.

In [3]:
# Limpieza de nombres de columnas
df.columns = (df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)
print("Esquema normalizado:", df.columns.tolist())

Esquema normalizado: ['movie_title', 'release_date', 'wikipedia_url', 'genre', 'director_1', 'director_2', 'cast_1', 'cast_2', 'cast_3', 'cast_4', 'cast_5', 'budget', 'revenue']


## 4. Aseguramiento de Tipos (Data Typing)
Hito: Conversión explícita a tipos numéricos (pd.to_numeric) y fechas, habilitando la capacidad de realizar cálculos aritméticos y análisis temporal.

In [4]:
# Conversión de métricas financieras para las columnas Budget y Revenue (originalmente cargadas como 'object')

# 4a. Limpieza de espacios en blanco en todas las columnas de texto
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 4b. Sanitización de métricas financieras: Eliminación de caracteres especiales.
for col in ['budget', 'revenue']:
    df[col] = df[col].astype(str).str.replace(r'[\$,]', '', regex=True)

df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')

# 4c. Conversión de fechas
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Verificación final de tipos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   movie_title    508 non-null    object        
 1   release_date   508 non-null    datetime64[ns]
 2   wikipedia_url  508 non-null    object        
 3   genre          508 non-null    object        
 4   director_1     508 non-null    object        
 5   director_2     41 non-null     object        
 6   cast_1         508 non-null    object        
 7   cast_2         503 non-null    object        
 8   cast_3         485 non-null    object        
 9   cast_4         452 non-null    object        
 10  cast_5         389 non-null    object        
 11  budget         508 non-null    float64       
 12  revenue        508 non-null    float64       
dtypes: datetime64[ns](1), float64(2), object(10)
memory usage: 51.7+ KB


## 5. Control de calidad
Hito: Validación de valores únicos (distinct) en categorías como Genre para asegurar la consistencia semántica del dataset. Conteo de valores nulos por columnas. Busqueda de duplicados.

In [5]:
# 5a. Validación de consistencia en la columna de género
print("Categorías únicas detectadas en 'genre':")
print(df['genre'].unique())

# 5b. Verificación de nulos resultantes de la limpieza
print("\nConteo de valores nulos por columna:")
print(df.isnull().sum())

# 5c. Identificación de registros duplicados
print("\nTotal de filas duplicadas:")
print(df.duplicated().sum())

# 5d. Gestión de valores faltantes
df[['director_2', 'cast_2', 'cast_3', 'cast_4', 'cast_5']] = df[['director_2', 'cast_2', 'cast_3', 'cast_4', 'cast_5']].fillna('N/A')

# 5e. Outliers Check: Verifico valores extremos en Budget/Revenue
pd.options.display.float_format = '{:.2f}'.format
print(df[['budget', 'revenue']].describe())
# Los valores máximos son coherentes con producciones de alto presupuesto.
# No se detectan valores negativos ni errores de escala.


Categorías únicas detectadas en 'genre':
['Thriller' 'Action' 'Comedy' 'Biography' 'Drama' 'Crime' 'Adventure'
 'Horror' 'Sci-Fi' 'Romance' 'Musical' 'Fantasy' 'Mystery' 'Family'
 'Religious' 'Animation' 'Documentary']

Conteo de valores nulos por columna:
movie_title        0
release_date       0
wikipedia_url      0
genre              0
director_1         0
director_2       467
cast_1             0
cast_2             5
cast_3            23
cast_4            56
cast_5           119
budget             0
revenue            0
dtype: int64

Total de filas duplicadas:
0
            budget      revenue
count       508.00       508.00
mean   48871397.64 151983208.66
std    49198592.03 183255470.10
min     1000000.00   1000000.00
25%    14000000.00  31100000.00
50%    30000000.00  79350000.00
75%    65000000.00 203725000.00
max   250000000.00 970800000.00


## 6. Exportación del dataset procesado
Hito: Generación del archivo final Movie-Data-Clean.csv para la ingesta en el Data Warehouse.

In [6]:
df.to_csv('../data/processed/Movie-Data-Clean.csv', index=False)

print("✅ Pipeline finalizado. Archivo guardado en: ../data/processed/Movie-Data-Clean.csv")
df.head()

✅ Pipeline finalizado. Archivo guardado en: ../data/processed/Movie-Data-Clean.csv


,movie_title,release_date,wikipedia_url,genre,director_1,director_2,cast_1,cast_2,cast_3,cast_4,cast_5,budget,revenue
0,10 Cloverfield Lane,2016-03-08,https://en.wikipedia.org/wiki/10_Cloverfield_Lane,Thriller,Dan Trachtenberg,N/A,Mary Elizabeth Winstead,John Goodman,John Gallagher,N/A,N/A,15000000.00,108300000.00
1,13 Hours: The Secret Soldiers of Benghazi,2016-01-15,https://en.wikipedia.org/wiki/13_Hours:_The_Se...,Action,Michael Bay,N/A,James Badge Dale,John Krasinski,Toby Stephens,Pablo Schreiber,Max Martini,45000000.00,69400000.00
2,2 Guns,2013-08-02,https://en.wikipedia.org/wiki/2_Guns,Action,Baltasar Kormákur,N/A,Mark Wahlberg,Denzel Washington,Paula Patton,Bill Paxton,Edward James Olmos,61000000.00,131900000.00
3,21 Jump Street,2012-03-16,https://en.wikipedia.org/wiki/21_Jump_Street_(...,Comedy,Phil Lord,Chris Miller,Jonah Hill,Channing Tatum,Ice Cube,Brie Larson,Rob Riggle,55000000.00,201500000.00
4,22 Jump Street,2014-06-04,https://en.wikipedia.org/wiki/22_Jump_Street,Action,Phil Lord,Chris Miller,Channing Tatum,Jonah Hill,Ice Cube,N/A,N/A,84500000.00,331300000.00
